# Análise dos Boletins de Urna — Eleições 2026 (1º e 2º turnos)

Fonte: arquivos `bweb_*.csv` do TSE em `1o-turno/` e `2o-turno/` (um grupo por turno).

O notebook:
1. Converte os CSVs (≈35 GB no 1º turno) para um cache Parquet compacto usando **DuckDB** — feito só na primeira execução.
2. Mostra o **resultado das eleições** de cada turno (Presidente, Governador, Senador, Deputados).
3. Identifica **urnas com votos para um único candidato** em um cargo.
4. Mede a **variação de votos totais (comparecimento) por urna e por cidade** entre o 1º e o 2º turno e destaca as que variaram **mais de 10%**.

In [1]:
# Dependência: pip install duckdb pandas
import time
from pathlib import Path

import duckdb
import pandas as pd

BASE = Path.cwd()
GRUPOS = {1: BASE / '1o-turno', 2: BASE / '2o-turno'}
CACHE = BASE / 'cache'
SAIDA = BASE / 'resultados'
CACHE.mkdir(exist_ok=True)
SAIDA.mkdir(exist_ok=True)

LIMIAR_VARIACAO = 0.10          # 10%
CARGOS_CANDIDATO_UNICO = ['Presidente', 'Governador', 'Senador']  # inclua 'Deputado Federal'/'Deputado Estadual' se quiser
MIN_ELEITORES_CANDIDATO_UNICO = 0  # filtre urnas muito pequenas, se desejar

con = duckdb.connect()
con.execute("SET memory_limit='12GB'")
con.execute(f"SET temp_directory='{CACHE / 'tmp'}'")

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Cache Parquet

Para cada turno são gerados dois arquivos:
- `votos_tN.parquet` — votos por urna/cargo/votável;
- `secoes_tN.parquet` — uma linha por urna (seção) com aptos, comparecimento e abstenções.

Apague a pasta `cache/` para reprocessar os CSVs.

In [2]:
def gerar_cache(turno: int, pasta: Path):
    votos_pq = CACHE / f'votos_t{turno}.parquet'
    secoes_pq = CACHE / f'secoes_t{turno}.parquet'
    if votos_pq.exists() and secoes_pq.exists():
        print(f'Turno {turno}: cache já existe')
        return
    t0 = time.time()
    csvs = str(pasta / 'bweb_*.csv')
    con.execute(f'''
        CREATE OR REPLACE TEMP VIEW bruto AS
        SELECT
            SG_UF                                   AS uf,
            TRY_CAST(CD_MUNICIPIO AS INTEGER)       AS cd_municipio,
            NM_MUNICIPIO                            AS municipio,
            TRY_CAST(NR_ZONA AS INTEGER)            AS zona,
            TRY_CAST(NR_SECAO AS INTEGER)           AS secao,
            TRY_CAST(NR_LOCAL_VOTACAO AS INTEGER)   AS local_votacao,
            TRY_CAST(CD_CARGO_PERGUNTA AS INTEGER)  AS cd_cargo,
            DS_CARGO_PERGUNTA                       AS cargo,
            DS_TIPO_VOTAVEL                         AS tipo_votavel,
            TRY_CAST(NR_VOTAVEL AS INTEGER)         AS nr_votavel,
            NM_VOTAVEL                              AS candidato,
            SG_PARTIDO                              AS partido,
            TRY_CAST(QT_VOTOS AS INTEGER)           AS votos,
            TRY_CAST(QT_APTOS AS INTEGER)           AS aptos,
            TRY_CAST(QT_COMPARECIMENTO AS INTEGER)  AS comparecimento,
            TRY_CAST(QT_ABSTENCOES AS INTEGER)      AS abstencoes,
            DS_TIPO_URNA                            AS tipo_urna,
            NR_URNA_EFETIVADA                       AS nr_urna
        FROM read_csv('{csvs}', delim=';', quote='"', header=true,
                      encoding='latin-1', all_varchar=true, parallel=true)
        WHERE TRY_CAST(CD_CARGO_PERGUNTA AS INTEGER) > 0
    ''')
    # Uma única leitura dos CSVs; as colunas por seção repetem-se mas comprimem bem
    con.execute(f"COPY (SELECT * FROM bruto) TO '{votos_pq}' (FORMAT parquet, COMPRESSION zstd)")
    print(f'Turno {turno}: votos gravados em {time.time() - t0:,.0f}s')
    con.execute(f'''
        COPY (
            SELECT uf, cd_municipio, any_value(municipio) AS municipio, zona, secao,
                   any_value(local_votacao) AS local_votacao,
                   max(aptos) AS aptos, max(comparecimento) AS comparecimento,
                   max(abstencoes) AS abstencoes,
                   any_value(tipo_urna) AS tipo_urna, any_value(nr_urna) AS nr_urna
            FROM read_parquet('{votos_pq}')
            GROUP BY uf, cd_municipio, zona, secao
        ) TO '{secoes_pq}' (FORMAT parquet, COMPRESSION zstd)
    ''')
    print(f'Turno {turno}: seções gravadas em {time.time() - t0:,.0f}s')


for turno, pasta in GRUPOS.items():
    gerar_cache(turno, pasta)

for turno in GRUPOS:
    con.execute(f"CREATE OR REPLACE VIEW votos_t{turno} AS SELECT * FROM read_parquet('{CACHE}/votos_t{turno}.parquet')")
    con.execute(f"CREATE OR REPLACE VIEW secoes_t{turno} AS SELECT * FROM read_parquet('{CACHE}/secoes_t{turno}.parquet')")

q = lambda sql: con.sql(sql).df()

Turno 1: cache já existe
Turno 2: cache já existe


## 2. Visão geral por turno

In [3]:
visao = pd.concat([
    q(f'''
        SELECT {t} AS turno, count(*) AS urnas, count(DISTINCT uf || cd_municipio) AS municipios,
               sum(aptos) AS aptos, sum(comparecimento) AS comparecimento,
               sum(abstencoes) AS abstencoes,
               round(100.0 * sum(comparecimento) / sum(aptos), 2) AS pct_comparecimento
        FROM secoes_t{t}
    ''') for t in GRUPOS
])
visao

,turno,urnas,municipios,aptos,comparecimento,abstencoes,pct_comparecimento
0,1,499206,5716,"158,745,007.00","125,275,835.00","33,469,172.00",78.92
0,2,499205,5715,"158,744,977.00","128,236,074.00","30,508,903.00",80.78


## 3. Resultado das eleições

`resultado(turno, cargo)` soma os votos por candidato. Percentuais calculados sobre **votos válidos** (nominais + legenda).
Para Presidente o resultado é nacional (inclui exterior, UF `ZZ`); para os demais cargos é por UF.

In [4]:
def resultado(turno: int, cargo: str, por_uf: bool = True) -> pd.DataFrame:
    uf_col = 'uf, ' if por_uf else ''
    part = 'PARTITION BY uf' if por_uf else ''
    return q(f'''
        WITH t AS (
            SELECT {uf_col} nr_votavel, candidato, partido, sum(votos) AS votos
            FROM votos_t{turno}
            WHERE cargo = '{cargo}' AND tipo_votavel IN ('Nominal', 'Legenda')
            GROUP BY ALL
        )
        SELECT {uf_col} nr_votavel, candidato, partido, votos,
               round(100.0 * votos / sum(votos) OVER ({part}), 2) AS pct_validos,
               rank() OVER ({part} ORDER BY votos DESC) AS posicao
        FROM t
        ORDER BY {uf_col} votos DESC
    ''')


def resumo_brancos_nulos(turno: int, cargo: str) -> pd.DataFrame:
    return q(f'''
        SELECT tipo_votavel, sum(votos) AS votos,
               round(100.0 * sum(votos) / sum(sum(votos)) OVER (), 2) AS pct_total
        FROM votos_t{turno} WHERE cargo = '{cargo}'
        GROUP BY tipo_votavel ORDER BY votos DESC
    ''')


def vencedores_por_uf(turno: int, cargo: str, vagas: int = 1) -> pd.DataFrame:
    df = resultado(turno, cargo)
    return df[df.posicao <= max(vagas, 2)].reset_index(drop=True)

### 3.1 — 1º turno

In [5]:
print('PRESIDENTE — 1º turno (nacional)')
display(resultado(1, 'Presidente', por_uf=False))
display(resumo_brancos_nulos(1, 'Presidente'))

PRESIDENTE — 1º turno (nacional)


,nr_votavel,candidato,partido,votos,pct_validos,posicao
0,22,FLAVIO BOLSONARO,PL,"56,104,503.00",47.03,1
1,13,LULA,PT,"53,879,538.00",45.16,2
2,70,ESCRITOR AUGUSTO CURY,AVANTE,"3,448,569.00",2.89,3
3,14,RENAN SANTOS,MISSÃO,"2,675,887.00",2.24,4
4,55,RONALDO CAIADO,PSD,"2,605,148.00",2.18,5
5,30,ZEMA,NOVO,"326,488.00",0.27,6
6,80,SAMARA,UP,"122,911.00",0.10,7
7,16,HERTZ DIAS,PSTU,"43,103.00",0.04,8
8,27,CLARIANA BARAO,DC,"40,043.00",0.03,9
9,21,EDMILSON COSTA,PCB,"22,693.00",0.02,10


,tipo_votavel,votos,pct_total
0,Nominal,"119,306,034.00",95.23
1,Nulo,"3,669,003.00",2.93
2,Branco,"2,300,798.00",1.84


In [6]:
print('PRESIDENTE — 1º turno por UF (votos por candidato)')
pres1_uf = resultado(1, 'Presidente').pivot_table(index='uf', columns='candidato', values='pct_validos')
pres1_uf

PRESIDENTE — 1º turno por UF (votos por candidato)


candidato,CLARIANA BARAO,EDMILSON COSTA,ESCRITOR AUGUSTO CURY,FLAVIO BOLSONARO,HERTZ DIAS,LEONARDO AVALANCHE,LULA,RENAN SANTOS,RONALDO CAIADO,RUI COSTA PIMENTA,SAMARA,VETERINÁRIO WILSON GRASSI,ZEMA
uf,,,,,,,,,,,,,
AC,0.05,0.03,2.88,64.55,0.02,0.00,28.73,1.70,1.83,0.01,0.07,0.01,0.12
AL,0.03,0.01,2.33,40.45,0.02,0.00,54.73,1.48,0.78,0.01,0.07,0.01,0.06
AM,0.04,0.01,3.57,44.99,0.02,0.00,48.23,1.63,1.30,0.01,0.09,0.01,0.08
AP,0.05,0.01,4.10,45.66,0.03,0.00,45.71,2.44,1.79,0.01,0.11,0.01,0.07
BA,0.02,0.02,2.28,28.53,0.02,0.00,66.17,1.37,1.43,0.01,0.05,0.01,0.08
CE,0.02,0.01,2.50,31.27,0.03,0.00,63.28,1.92,0.81,0.01,0.06,0.01,0.08
DF,0.04,0.02,2.59,51.31,0.05,0.00,38.11,2.63,4.80,0.02,0.13,0.02,0.28
ES,0.04,0.02,2.89,54.78,0.03,0.01,37.76,2.36,1.74,0.01,0.09,0.01,0.26
GO,0.01,0.01,1.41,53.60,0.02,0.00,31.06,1.41,12.33,0.01,0.05,0.01,0.07


In [7]:
print('GOVERNADOR — 1º turno: dois primeiros colocados por UF')
vencedores_por_uf(1, 'Governador')

GOVERNADOR — 1º turno: dois primeiros colocados por UF


,uf,nr_votavel,candidato,partido,votos,pct_validos,posicao
0,AC,11,MAILZA ASSIS,PP,"218,760.00",49.76,1
1,AC,10,ALAN RICK,REPUBLICANOS,"141,859.00",32.27,2
2,AL,45,JHC,PSDB,"892,484.00",52.20,1
3,AL,15,RENAN FILHO,MDB,"809,796.00",47.37,2
4,AM,55,OMAR AZIZ,PSD,"841,846.00",40.63,1
5,AM,22,PROFESSORA MARIA DO CARMO,PL,"507,335.00",24.49,2
6,AP,55,DR. FURLAN,PSD,"281,595.00",62.75,1
7,AP,44,CLÉCIO,UNIÃO,"165,900.00",36.97,2
8,BA,13,JERÔNIMO RODRIGUES,PT,"4,626,509.00",55.80,1
9,BA,44,ACM NETO,UNIÃO,"3,625,227.00",43.72,2


In [8]:
print('SENADOR — 1º turno: três primeiros colocados por UF')
sen = resultado(1, 'Senador')
sen[sen.posicao <= 3].reset_index(drop=True)

SENADOR — 1º turno: três primeiros colocados por UF


,uf,nr_votavel,candidato,partido,votos,pct_validos,posicao
0,AC,222,MARCIO BITTAR,PL,"223,972.00",25.59,1
1,AC,100,MARA ROCHA,REPUBLICANOS,"157,104.00",17.95,2
2,AC,111,GLADSON CAMELÍ,PP,"145,244.00",16.59,3
3,AL,111,ARTHUR LIRA,PP,"948,703.00",28.87,1
4,AL,456,MARINA JHC,PSDB,"926,860.00",28.20,2
5,AL,151,RENAN,MDB,"731,045.00",22.24,3
6,AM,155,EDUARDO BRAGA,MDB,"1,240,471.00",31.81,1
7,AM,455,PLINIO VALÉRIO,PSDB,"948,793.00",24.33,2
8,AM,222,CAPITÃO ALBERTO NETO,PL,"909,155.00",23.32,3
9,AP,200,RAYSSA FURLAN,PODE,"264,799.00",29.98,1


In [9]:
for cargo in ['Deputado Federal', 'Deputado Estadual']:
    print(f'{cargo.upper()} — 1º turno: votos válidos (nominal + legenda) por partido, nacional')
    display(q(f'''
        SELECT partido, sum(votos) AS votos,
               round(100.0 * sum(votos) / sum(sum(votos)) OVER (), 2) AS pct_validos
        FROM votos_t1
        WHERE cargo = '{cargo}' AND tipo_votavel IN ('Nominal', 'Legenda')
        GROUP BY partido ORDER BY votos DESC
    '''))

DEPUTADO FEDERAL — 1º turno: votos válidos (nominal + legenda) por partido, nacional


,partido,votos,pct_validos
0,PL,"25,830,609.00",22.64
1,PT,"14,798,202.00",12.97
2,PSD,"9,403,436.00",8.24
3,REPUBLICANOS,"7,987,717.00",7.00
4,MDB,"7,924,588.00",6.94
5,UNIÃO,"7,877,391.00",6.90
6,PP,"7,620,142.00",6.68
7,PODE,"5,900,293.00",5.17
8,PSOL,"5,312,186.00",4.66
9,PSB,"5,013,154.00",4.39


DEPUTADO ESTADUAL — 1º turno: votos válidos (nominal + legenda) por partido, nacional


,partido,votos,pct_validos
0,PL,"22,522,692.00",20.17
1,PT,"15,454,635.00",13.84
2,PSD,"11,410,805.00",10.22
3,MDB,"10,203,697.00",9.14
4,REPUBLICANOS,"7,116,567.00",6.37
5,PP,"6,847,675.00",6.13
6,UNIÃO,"6,309,668.00",5.65
7,PODE,"4,756,298.00",4.26
8,PSB,"4,548,694.00",4.07
9,PSOL,"4,150,226.00",3.72


In [10]:
print('DEPUTADO FEDERAL — 10 mais votados por UF')
dep = resultado(1, 'Deputado Federal')
dep[(dep.posicao <= 10) & (dep.candidato != dep.partido)].reset_index(drop=True)

DEPUTADO FEDERAL — 10 mais votados por UF


,uf,nr_votavel,candidato,partido,votos,pct_validos,posicao
0,AC,1101,SOCORRO NERI,PP,"39,587.00",8.45,1
1,AC,4444,DR. FÁBIO RUEDA,UNIÃO,"33,203.00",7.08,2
2,AC,4422,CORONEL ULYSSES,UNIÃO,"33,155.00",7.07,3
3,AC,1533,PEDRO LONGO,MDB,"26,263.00",5.60,4
4,AC,1010,ROBERTO DUARTE,REPUBLICANOS,"24,854.00",5.30,5
...,...,...,...,...,...,...,...
265,TO,2222,FILIPE MARTINS,PL,"64,068.00",7.06,6
266,TO,2022,SANDOVAL CARDOSO,PODE,"61,747.00",6.81,7
267,TO,1011,ALFREDO JÚNIOR,REPUBLICANOS,"48,253.00",5.32,8
268,TO,4422,CORONEL BARBOSA,UNIÃO,"28,544.00",3.15,9


### 3.2 — 2º turno

In [11]:
print('PRESIDENTE — 2º turno (nacional)')
display(resultado(2, 'Presidente', por_uf=False))
display(resumo_brancos_nulos(2, 'Presidente'))

PRESIDENTE — 2º turno (nacional)


,nr_votavel,candidato,partido,votos,pct_validos,posicao
0,13,LULA,PT,"64,256,232.00",51.79,1
1,22,FLAVIO BOLSONARO,PL,"59,826,041.00",48.21,2


,tipo_votavel,votos,pct_total
0,Nominal,"124,082,273.00",96.76
1,Nulo,"2,723,305.00",2.12
2,Branco,"1,430,496.00",1.12


In [12]:
print('PRESIDENTE — 2º turno por UF (% válidos)')
resultado(2, 'Presidente').pivot_table(index='uf', columns='candidato', values='pct_validos')

PRESIDENTE — 2º turno por UF (% válidos)


candidato,FLAVIO BOLSONARO,LULA
uf,,
AC,66.57,33.43
AL,39.23,60.77
AM,45.98,54.02
AP,47.12,52.88
BA,28.69,71.31
CE,31.07,68.93
DF,54.28,45.72
ES,56.37,43.63
GO,59.66,40.34


In [13]:
print('GOVERNADOR — 2º turno por UF')
resultado(2, 'Governador')

GOVERNADOR — 2º turno por UF


,uf,nr_votavel,candidato,partido,votos,pct_validos,posicao
0,AC,11,MAILZA ASSIS,PP,"272,635.00",66.21,1
1,AC,10,ALAN RICK,REPUBLICANOS,"139,108.00",33.79,2
2,AM,55,OMAR AZIZ,PSD,"1,273,975.00",61.42,1
3,AM,22,PROFESSORA MARIA DO CARMO,PL,"800,328.00",38.58,2
4,DF,11,CELINA LEÃO,PP,"987,142.00",57.18,1
5,DF,13,LEANDRO GRASS,PT,"739,130.00",42.82,2
6,ES,10,LORENZO PAZOLINI,REPUBLICANOS,"1,151,818.00",52.17,1
7,ES,15,RICARDO FERRAÇO,MDB,"1,056,119.00",47.83,2
8,RN,13,CADU DE LULA,PT,"1,073,024.00",50.77,1
9,RN,44,ALLYSON,UNIÃO,"1,040,399.00",49.23,2


## 4. Urnas com votos para um único candidato

Para cada urna e cargo, contamos quantos candidatos (votos **nominais**) receberam ao menos 1 voto.
Se apenas **um** candidato recebeu votos, a urna é marcada. Brancos e nulos são mostrados à parte.
Ajuste `CARGOS_CANDIDATO_UNICO` e `MIN_ELEITORES_CANDIDATO_UNICO` na célula de configuração.

In [14]:
cargos_sql = ', '.join(f"'{c}'" for c in CARGOS_CANDIDATO_UNICO)


def candidato_unico(turno: int) -> pd.DataFrame:
    return q(f'''
        WITH por_urna AS (
            SELECT uf, cd_municipio, zona, secao, cargo,
                   count(*) FILTER (WHERE tipo_votavel = 'Nominal' AND votos > 0) AS candidatos_com_voto,
                   sum(votos) FILTER (WHERE tipo_votavel = 'Nominal')  AS votos_nominais,
                   coalesce(sum(votos) FILTER (WHERE tipo_votavel = 'Branco'), 0) AS brancos,
                   coalesce(sum(votos) FILTER (WHERE tipo_votavel = 'Nulo'), 0)   AS nulos,
                   arg_max(candidato, votos) FILTER (WHERE tipo_votavel = 'Nominal') AS candidato,
                   arg_max(partido, votos)   FILTER (WHERE tipo_votavel = 'Nominal') AS partido
            FROM votos_t{turno}
            WHERE cargo IN ({cargos_sql})
            GROUP BY ALL
        )
        SELECT {turno} AS turno, p.uf, s.municipio, p.zona, p.secao, s.local_votacao, p.cargo,
               p.candidato, p.partido, p.votos_nominais, p.brancos, p.nulos,
               s.aptos, s.comparecimento, s.nr_urna
        FROM por_urna p
        JOIN secoes_t{turno} s USING (uf, cd_municipio, zona, secao)
        WHERE p.candidatos_com_voto = 1
          AND s.comparecimento >= {MIN_ELEITORES_CANDIDATO_UNICO}
        ORDER BY p.cargo, s.comparecimento DESC
    ''')


unico = pd.concat([candidato_unico(t) for t in GRUPOS], ignore_index=True)
print(f'{len(unico):,} ocorrências (urna × cargo) com votos para um único candidato')
unico.groupby(['turno', 'cargo']).agg(urnas=('secao', 'size'),
                                      votos_nominais=('votos_nominais', 'sum'),
                                      maior_urna=('comparecimento', 'max'))

204 ocorrências (urna × cargo) com votos para um único candidato


urnas  votos_nominais  maior_urna
turno cargo                                        
1     Governador     21        1,613.00         297
      Presidente     40        3,764.00         283
      Senador         1          108.00         238
2     Governador     32        5,912.00         365
      Presidente    110       15,156.00         365

In [15]:
print('Distribuição por UF / município')
(unico.groupby(['turno', 'cargo', 'uf', 'municipio'])
      .agg(urnas=('secao', 'size'), votos=('votos_nominais', 'sum'),
           candidatos=('candidato', lambda s: ', '.join(sorted(s.dropna().unique()))))
      .sort_values('urnas', ascending=False))

Distribuição por UF / município


urnas    votos                candidatos
turno cargo      uf municipio                                                          
2     Presidente RR UIRAMUTÃ                       7 1,163.00                      LULA
                 AM ATALAIA DO NORTE               5 1,027.00                      LULA
                    SÃO GABRIEL DA CACHOEIRA       5   954.00                      LULA
      Governador AM ATALAIA DO NORTE               5 1,023.00                 OMAR AZIZ
      Presidente MA TURIAÇU                        4   485.00                      LULA
      Governador AC JORDÃO                         3   386.00              MAILZA ASSIS
                 AM BARREIRINHA                    3   685.00                 OMAR AZIZ
                    BENJAMIN CONSTANT              3   677.00                 OMAR AZIZ
                    SÃO GABRIEL DA CACHOEIRA       3   523.00                 OMAR AZIZ
                    SANTA ISABEL DO RIO NEGRO      3   554.00                 OMAR AZIZ
      Presidente PA ORIXIMINÁ                      3   320.00                      LULA
1     Presidente RR UIRAMUTÃ                       3   428.00                      LULA
2     Presidente TO GOIATINS                       3   444.00                      LULA
                 AM SANTA ISABEL DO RIO NEGRO      3   537.00                      LULA
                    TABATINGA                      2   692.00                      LULA
                 MA SERRANO DO MARANHÃO            2   157.00                      LULA
                    SANTA HELENA                   2   109.00                      LULA
1     Presidente PA NOVO REPARTIMENTO              2   278.00                      LULA
2     Presidente MA PRIMEIRA CRUZ                  2    84.00                      LULA
                 AC JORDÃO                         2   249.00                      LULA
      Governador AM AMATURÁ                        2   272.00                 OMAR AZIZ
      Presidente PA ALTAMIRA                       2    87.00                      LULA
                 MA ALCÂNTARA                      2   123.00                      LULA
      Governador AM SÃO PAULO DE OLIVENÇA          2   491.00                 OMAR AZIZ
      Presidente AM BENJAMIN CONSTANT              2   469.00                      LULA
                    MAUÉS                          2   456.00                      LULA
                    PARINTINS                      2   308.00                      LULA
                    SANTO ANTÔNIO DO IÇÁ           2   644.00                      LULA
1     Presidente PA ALTAMIRA                       2    75.00                      LULA
2     Presidente AM SÃO PAULO DE OLIVENÇA          2   542.00                      LULA
                 RS VIAMÃO                         2   104.00                      LULA
                 TO TOCANTINÓPOLIS                 2   214.00                      LULA
1     Presidente MA SANTA HELENA                   2   103.00                      LULA
      Governador PA ALTAMIRA                       2    70.00              HANA GHASSAN
                    NOVO REPARTIMENTO              2   273.00              HANA GHASSAN
                    PARAUAPEBAS                    2   271.00              HANA GHASSAN
2     Presidente PA PARAUAPEBAS                    2   409.00                      LULA
                    SÃO FÉLIX DO XINGU             2   418.00                      LULA
1     Presidente AM SÃO GABRIEL DA CACHOEIRA       2   373.00                      LULA
      Governador PA SÃO FÉLIX DO XINGU             2   277.00  DR. DANIEL, HANA GHASSAN
2     Presidente PA AVEIRO                         1   106.00                      LULA
                 MA BARREIRINHAS                   1    95.00                      LULA
                 RR CAROEBE                        1    44.00                      LULA
                 RO PORTO VELHO                    1   104.00                      LULA

In [16]:
print('Detalhe — urnas com candidato único (maiores primeiro)')
unico.sort_values('comparecimento', ascending=False).reset_index(drop=True)

Detalhe — urnas com candidato único (maiores primeiro)


,turno,uf,municipio,zona,secao,local_votacao,cargo,candidato,partido,votos_nominais,brancos,nulos,aptos,comparecimento,nr_urna
0,2,AM,TABATINGA,36,61,1201,Governador,OMAR AZIZ,PSD,347.00,0.00,18.00,400,365,2332683
1,2,AM,TABATINGA,36,61,1201,Presidente,LULA,PT,354.00,0.00,11.00,400,365,2332683
2,2,AM,TABATINGA,36,96,1260,Presidente,LULA,PT,338.00,0.00,9.00,388,347,2352217
3,2,AM,MAUÉS,5,176,1406,Presidente,LULA,PT,333.00,0.00,4.00,382,337,2341225
4,2,AM,ATALAIA DO NORTE,42,29,1112,Presidente,LULA,PT,325.00,1.00,9.00,441,335,2354152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199,1,PB,SAPÉ,4,290,1597,Governador,LUCAS RIBEIRO,PP,5.00,3.00,5.00,21,13,2455215
200,1,BA,CAMAÇARI,171,409,1589,Presidente,LULA,PT,10.00,0.00,0.00,24,10,1754593
201,2,PB,SAPÉ,4,290,1597,Presidente,LULA,PT,10.00,0.00,0.00,21,10,2455215
202,1,ZZ,PORTO PRÍNCIPE,1,353,1015,Presidente,LULA,PT,5.00,0.00,0.00,33,5,1808076


## 5. Variação de votos totais por urna e por cidade (1º × 2º turno)

“Votos totais” = **comparecimento** da urna (todos os eleitores que votaram, incluindo brancos e nulos).
A chave da urna é `UF + município + zona + seção`.

- **5.1** — cidades cuja soma de votos variou mais de 10% entre os turnos;
- **5.2** — urnas individuais que variaram mais de 10%;
- **5.3** — urnas presentes em apenas um dos turnos;
- **5.4** (complementar) — dispersão do nº de votos entre as urnas de uma mesma cidade, por turno.

In [17]:
_ = con.execute('''
    CREATE OR REPLACE TEMP TABLE var_urna AS
    SELECT coalesce(a.uf, b.uf) AS uf, coalesce(a.cd_municipio, b.cd_municipio) AS cd_municipio,
           coalesce(a.municipio, b.municipio) AS municipio,
           coalesce(a.zona, b.zona) AS zona, coalesce(a.secao, b.secao) AS secao,
           a.aptos AS aptos_t1, b.aptos AS aptos_t2,
           a.comparecimento AS votos_t1, b.comparecimento AS votos_t2,
           b.comparecimento - a.comparecimento AS diferenca,
           (b.comparecimento - a.comparecimento) / nullif(a.comparecimento, 0) AS variacao,
           a.nr_urna AS urna_t1, b.nr_urna AS urna_t2
    FROM secoes_t1 a FULL OUTER JOIN secoes_t2 b USING (uf, cd_municipio, zona, secao)
''')

var_cidade = q(f'''
    SELECT uf, cd_municipio, municipio,
           count(*) FILTER (WHERE votos_t1 IS NOT NULL AND votos_t2 IS NOT NULL) AS urnas,
           sum(votos_t1) AS votos_t1, sum(votos_t2) AS votos_t2,
           sum(votos_t2) - sum(votos_t1) AS diferenca,
           round(100.0 * (sum(votos_t2) - sum(votos_t1)) / nullif(sum(votos_t1), 0), 2) AS variacao_pct,
           count(*) FILTER (WHERE abs(variacao) > {LIMIAR_VARIACAO}) AS urnas_var_acima_10pct,
           round(100.0 * count(*) FILTER (WHERE abs(variacao) > {LIMIAR_VARIACAO})
                 / count(*) FILTER (WHERE variacao IS NOT NULL), 2) AS pct_urnas_acima_10pct,
           round(100.0 * min(variacao), 2) AS menor_var_urna_pct,
           round(100.0 * max(variacao), 2) AS maior_var_urna_pct
    FROM var_urna
    GROUP BY ALL
    ORDER BY abs(variacao_pct) DESC
''')
print(f'{len(var_cidade):,} municípios comparados')
var_cidade.describe()

5,716 municípios comparados


,cd_municipio,urnas,votos_t1,votos_t2,diferenca,variacao_pct,urnas_var_acima_10pct,pct_urnas_acima_10pct,menor_var_urna_pct,maior_var_urna_pct
count,"5,716.00","5,716.00","5,716.00","5,715.00","5,715.00","5,715.00","5,716.00","5,715.00","5,715.00","5,715.00"
mean,"51,054.06",87.33,"21,916.70","22,438.51",517.98,1.85,2.20,3.97,-4.69,8.49
std,"29,236.50",459.74,"124,376.42","128,881.66","4,716.81",3.66,13.03,12.62,6.30,5.28
min,19.00,0.00,2.00,7.00,"-19,220.00",-28.76,0.00,0.00,-89.41,-15.61
25%,"24,989.50",17.00,"3,747.00","3,809.00",24.00,0.67,0.00,0.00,-6.32,5.65
50%,"49,985.50",31.00,"7,110.50","7,226.00",137.00,2.38,0.00,0.00,-3.36,8.11
75%,"78,658.00",65.00,"15,087.50","15,464.50",381.50,3.52,1.00,2.86,-1.47,10.80
max,"99,511.00","26,683.00","7,006,411.00","7,284,071.00","277,660.00",60.00,569.00,100.00,60.00,94.74


### 5.1 Cidades com variação total > 10% entre os turnos

In [18]:
cidades_10 = var_cidade[var_cidade.variacao_pct.abs() > LIMIAR_VARIACAO * 100].reset_index(drop=True)
print(f'{len(cidades_10):,} municípios com |variação| > {LIMIAR_VARIACAO:.0%}')
cidades_10

114 municípios com |variação| > 10%


,uf,cd_municipio,municipio,urnas,votos_t1,votos_t2,diferenca,variacao_pct,urnas_var_acima_10pct,pct_urnas_acima_10pct,menor_var_urna_pct,maior_var_urna_pct
0,ZZ,30333,PORTO PRÍNCIPE,1,5.00,8.00,3.00,60.00,1,100.00,60.00,60.00
1,ZZ,30805,WELLINGTON,5,339.00,526.00,187.00,55.16,5,100.00,20.00,94.74
2,ZZ,30295,PASO LOS LIBRES,1,5.00,7.00,2.00,40.00,1,100.00,40.00,40.00
3,ZZ,30848,XANGAI,1,197.00,264.00,67.00,34.01,1,100.00,34.01,34.01
4,AP,6041,ITAUBAL,31,"7,354.00","5,239.00","-2,115.00",-28.76,31,100.00,-47.18,-14.97
5,AP,6068,CUTIAS,23,"5,191.00","3,758.00","-1,433.00",-27.61,23,100.00,-46.93,-15.61
6,AC,1066,PORTO WALTER,34,"6,520.00","4,792.00","-1,728.00",-26.50,34,100.00,-48.40,-12.18
7,ZZ,30325,PORT OF SPAIN,1,19.00,24.00,5.00,26.32,1,100.00,26.32,26.32
8,ZZ,30740,VARSÓVIA,1,291.00,363.00,72.00,24.74,1,100.00,24.74,24.74
9,AC,1040,MARECHAL THAUMATURGO,47,"9,352.00","7,051.00","-2,301.00",-24.60,42,89.36,-50.22,-2.74


In [19]:
print('Cidades com maior proporção de urnas que variaram > 10%')
var_cidade[var_cidade.urnas >= 5].sort_values('pct_urnas_acima_10pct', ascending=False).head(50).reset_index(drop=True)

Cidades com maior proporção de urnas que variaram > 10%


,uf,cd_municipio,municipio,urnas,votos_t1,votos_t2,diferenca,variacao_pct,urnas_var_acima_10pct,pct_urnas_acima_10pct,menor_var_urna_pct,maior_var_urna_pct
0,ZZ,30805,WELLINGTON,5,339.00,526.00,187.00,55.16,5,100.00,20.00,94.74
1,ZZ,30619,TEL AVIV,5,730.00,902.00,172.00,23.56,5,100.00,19.05,31.58
2,AP,6106,SERRA DO NAVIO,12,"3,040.00","2,499.00",-541.00,-17.80,12,100.00,-22.71,-12.58
3,AP,6017,AMAPÁ,27,"6,262.00","4,971.00","-1,291.00",-20.62,27,100.00,-27.75,-13.39
4,AP,6033,CALÇOENE,30,"7,009.00","5,773.00","-1,236.00",-17.63,30,100.00,-31.15,-10.68
5,ZZ,30902,HARTFORD,12,"1,951.00","2,378.00",427.00,21.89,12,100.00,12.21,31.94
6,ZZ,30864,ZURIQUE,19,"6,001.00","7,086.00","1,085.00",18.08,19,100.00,11.47,29.04
7,ZZ,29351,BARCELONA,22,"5,185.00","6,381.00","1,196.00",23.07,22,100.00,12.90,36.84
8,AP,6114,FERREIRA GOMES,26,"6,260.00","4,764.00","-1,496.00",-23.90,26,100.00,-33.87,-13.33
9,AP,6009,PRACUÚBA,16,"3,313.00","2,510.00",-803.00,-24.24,16,100.00,-43.97,-12.20


### 5.2 Urnas com variação > 10% entre os turnos

In [20]:
urnas_10 = q(f'''
    SELECT uf, municipio, zona, secao, aptos_t1, aptos_t2, votos_t1, votos_t2, diferenca,
           round(100.0 * variacao, 2) AS variacao_pct, urna_t1, urna_t2,
           urna_t1 <> urna_t2 AS urna_substituida
    FROM var_urna
    WHERE abs(variacao) > {LIMIAR_VARIACAO}
    ORDER BY abs(variacao) DESC
''')
print(f'{len(urnas_10):,} urnas com |variação| > {LIMIAR_VARIACAO:.0%} '
      f'(de {q("SELECT count(*) FROM var_urna WHERE variacao IS NOT NULL").iloc[0, 0]:,} comparáveis)')
display(urnas_10.groupby('uf').size().rename('urnas').to_frame().T)
urnas_10.head(200)

12,580 urnas com |variação| > 10% (de 499,205 comparáveis)


uf,AC,AL,AM,AP,BA,CE,DF,ES,GO,MA,MG,MS,MT,PA,PB,PE,PI,PR,RJ,RN,RO,RR,RS,SC,SE,SP,TO,ZZ
urnas,599,147,537,720,1607,223,9,59,109,648,1135,400,100,961,82,522,139,408,656,126,22,442,302,56,36,1684,41,810


,uf,municipio,zona,secao,aptos_t1,aptos_t2,votos_t1,votos_t2,diferenca,variacao_pct,urna_t1,urna_t2,urna_substituida
0,ZZ,WELLINGTON,1,1693,699,699,38,74,36,94.74,1664895,1664895,False
1,PR,INÁCIO MARTINS,34,133,215,215,170,18,-152,-89.41,2060731,2060731,False
2,ZZ,WELLINGTON,1,1088,696,697,40,73,33,82.50,1652005,1652005,False
3,RJ,MANGARATIBA,54,140,131,131,89,18,-71,-79.78,2350559,2350559,False
4,PA,ALTAMIRA,18,519,54,54,21,35,14,66.67,2432252,2432252,False
5,PA,PARAUAPEBAS,106,330,325,325,128,212,84,65.63,2191815,2191815,False
6,RR,NORMANDIA,9,77,134,134,73,26,-47,-64.38,2260555,2260555,False
7,ZZ,WELLINGTON,1,1690,700,700,101,163,62,61.39,1663973,1663973,False
8,ZZ,PORTO PRÍNCIPE,1,353,33,33,5,8,3,60.00,1808076,1808076,False
9,AC,SENA MADUREIRA,3,148,216,216,152,65,-87,-57.24,2047901,2047901,False


In [21]:
print('Urnas com variação > 10% em que o nº de eleitores aptos também mudou (possível remanejamento de seção)')
urnas_10[urnas_10.aptos_t1 != urnas_10.aptos_t2].reset_index(drop=True)

Urnas com variação > 10% em que o nº de eleitores aptos também mudou (possível remanejamento de seção)


,uf,municipio,zona,secao,aptos_t1,aptos_t2,votos_t1,votos_t2,diferenca,variacao_pct,urna_t1,urna_t2,urna_substituida
0,ZZ,WELLINGTON,1,1088,696,697,40,73,33,82.50,1652005,1652005,False
1,AC,PORTO WALTER,4,429,234,233,188,97,-91,-48.40,2260491,2260491,False
2,SP,DRACENA,149,119,39,27,34,18,-16,-47.06,1789952,1789952,False
3,AC,MARECHAL THAUMATURGO,4,294,271,270,237,126,-111,-46.84,2256772,2256772,False
4,AC,MARECHAL THAUMATURGO,4,162,262,260,225,124,-101,-44.89,2273101,2273101,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1138,BA,MARAÚ,78,144,366,367,279,307,28,10.04,1809729,1809729,False
1139,RJ,RIO DE JANEIRO,123,139,450,452,289,318,29,10.03,2131058,2131058,False
1140,BA,ITAPETINGA,140,71,393,392,299,329,30,10.03,2077043,2077043,False
1141,PE,GARANHUNS,56,216,387,388,309,340,31,10.03,1773689,1773689,False


### 5.3 Urnas presentes em só um turno

In [22]:
q('''
    SELECT CASE WHEN votos_t1 IS NULL THEN 'só 2º turno' ELSE 'só 1º turno' END AS situacao,
           uf, municipio, zona, secao, coalesce(votos_t1, votos_t2) AS votos
    FROM var_urna
    WHERE votos_t1 IS NULL OR votos_t2 IS NULL
    ORDER BY situacao, uf, municipio, zona, secao
''')

,situacao,uf,municipio,zona,secao,votos
0,só 1º turno,ZZ,TRÍPOLI,1,443,2


### 5.4 (Complementar) Dispersão de votos por urna dentro de cada cidade

Coeficiente de variação (desvio padrão / média) do comparecimento por urna em cada município e turno.

In [23]:
dispersao = q('''
    SELECT turno, uf, municipio, count(*) AS urnas,
           round(avg(comparecimento), 1) AS media_votos_urna,
           round(stddev_samp(comparecimento), 1) AS desvio,
           round(100.0 * stddev_samp(comparecimento) / avg(comparecimento), 2) AS cv_pct,
           min(comparecimento) AS min_votos, max(comparecimento) AS max_votos
    FROM (SELECT 1 AS turno, * FROM secoes_t1 UNION ALL BY NAME SELECT 2 AS turno, * FROM secoes_t2)
    GROUP BY ALL
    HAVING count(*) >= 5
    ORDER BY cv_pct DESC
''')
dispersao.pivot_table(index=['uf', 'municipio'], columns='turno', values='cv_pct') \
         .rename(columns={1: 'cv_pct_t1', 2: 'cv_pct_t2'}) \
         .assign(delta=lambda d: d.cv_pct_t2 - d.cv_pct_t1) \
         .sort_values('delta', key=abs, ascending=False).head(50)

,turno,cv_pct_t1,cv_pct_t2,delta
uf,municipio,,,
PR,INÁCIO MARTINS,32.74,40.27,7.53
AP,CUTIAS,17.73,24.62,6.89
GO,ANHANGÜERA,26.64,33.27,6.63
AP,ITAUBAL,21.42,27.90,6.48
AL,OLHO D'ÁGUA GRANDE,6.05,11.83,5.78
AC,PORTO WALTER,21.91,27.29,5.38
MS,PARANHOS,22.94,17.81,-5.13
SP,SANTO EXPEDITO,15.63,20.18,4.55
ZZ,SÃO FRANCISCO,34.72,30.38,-4.34


## 6. Testes de indícios de manipulação (eleição para Presidente)

Testes estatísticos usuais de *election forensics*. Eles detectam padrões típicos de fraude na contagem
(inclusão de votos, números inventados, urnas destoantes), mas **não provam ausência de fraude** nem detectam
compra de votos ou coação.

| Teste | O que indicaria problema |
|---|---|
| 6.1 Candidato único × local de votação | urna 100% para um candidato cercada de urnas com votação diferente |
| 6.2 Distribuição do % por urna | pico artificial em 0% ou 100% |
| 6.3 Comparecimento ≥ 97% | muitas urnas com comparecimento quase total e voto concentrado |
| 6.4 Comparecimento × % do candidato | correlação positiva forte / nuvem de urnas indo para (100%, 100%) |
| 6.5 Último dígito | frequência de cada dígito muito diferente de 10% |
| 6.6 Variação 1º→2º turno × voto | urnas com variação > 10% com mudança de voto diferente das demais |
| 6.7 Urnas substituídas | resultado diferente nas urnas trocadas entre os turnos |

In [24]:
_ = con.execute('''
    CREATE OR REPLACE TEMP TABLE pres AS
    SELECT turno, uf, cd_municipio, zona, secao, any_value(local_votacao) AS local_votacao,
           coalesce(sum(votos) FILTER (WHERE nr_votavel = 13), 0) AS lula,
           coalesce(sum(votos) FILTER (WHERE nr_votavel = 22), 0) AS flavio,
           sum(votos) FILTER (WHERE tipo_votavel = 'Nominal') AS validos,
           max(aptos) AS aptos, max(comparecimento) AS comparecimento, any_value(nr_urna) AS nr_urna
    FROM (SELECT 1 AS turno, * FROM votos_t1 UNION ALL BY NAME SELECT 2 AS turno, * FROM votos_t2)
    WHERE cargo = 'Presidente'
    GROUP BY ALL
''')

### 6.1 Urnas de candidato único × município e local de votação (2º turno, ≥ 30 votos válidos)

Se as demais urnas do mesmo local também votaram ~100% no mesmo candidato, o padrão é do local
(típico de aldeias indígenas e comunidades isoladas), não de uma urna isolada.

In [25]:
unico_local = q('''
    WITH mun AS (SELECT uf, cd_municipio, 100.0 * sum(lula) / sum(validos) AS pct_lula_municipio
                 FROM pres WHERE turno = 2 GROUP BY ALL),
         loc AS (SELECT uf, cd_municipio, zona, local_votacao, count(*) AS urnas_no_local,
                        100.0 * sum(lula) / sum(validos) AS pct_lula_local
                 FROM pres WHERE turno = 2 GROUP BY ALL)
    SELECT p.uf, s.municipio, p.zona, p.secao, p.local_votacao, p.validos,
           CASE WHEN p.lula = p.validos THEN 'LULA' ELSE 'FLAVIO BOLSONARO' END AS candidato,
           round(pct_lula_municipio, 1) AS pct_lula_municipio,
           round(pct_lula_local, 1) AS pct_lula_local, urnas_no_local
    FROM pres p
    JOIN (SELECT uf, cd_municipio, zona, secao, municipio FROM secoes_t2) s USING (uf, cd_municipio, zona, secao)
    JOIN mun USING (uf, cd_municipio)
    JOIN loc USING (uf, cd_municipio, zona, local_votacao)
    WHERE p.turno = 2 AND p.validos >= 30 AND (p.lula = p.validos OR p.flavio = p.validos)
    ORDER BY abs(pct_lula_local - pct_lula_municipio) DESC
''')
display(unico_local.groupby('candidato')[['pct_lula_municipio', 'pct_lula_local']].describe().T)
unico_local

candidato                 FLAVIO BOLSONARO   LULA
pct_lula_municipio count              4.00 103.00
                   mean              39.53  67.19
                   std               25.25  17.60
                   min               13.90  18.20
                   25%               21.70  55.70
                   50%               37.25  73.80
                   75%               55.08  80.00
                   max               69.70  90.90
pct_lula_local     count              4.00 103.00
                   mean               0.17  99.88
                   std                0.35   0.28
                   min                0.00  98.50
                   25%                0.00 100.00
                   50%                0.00 100.00
                   75%                0.17 100.00
                   max                0.70 100.00

,uf,municipio,zona,secao,local_votacao,validos,candidato,pct_lula_municipio,pct_lula_local,urnas_no_local
0,MT,GUARANTÃ DO NORTE,44,284,1171,145.00,LULA,18.20,100.00,1
1,RR,CAROEBE,4,141,1163,44.00,LULA,20.80,100.00,1
2,MT,CONFRESA,28,200,1120,309.00,LULA,25.80,99.80,2
3,PA,SÃO FÉLIX DO XINGU,53,218,1350,273.00,LULA,27.60,100.00,1
4,PA,SÃO FÉLIX DO XINGU,53,217,1341,145.00,LULA,27.60,100.00,1
5,AC,CRUZEIRO DO SUL,4,457,2291,137.00,LULA,28.90,99.20,2
6,PA,CHAVES,17,41,1252,48.00,FLAVIO BOLSONARO,69.70,0.00,1
7,MT,PORTO ALEGRE DO NORTE,28,191,1090,61.00,LULA,34.00,100.00,1
8,PA,ALTAMIRA,18,519,3000,35.00,LULA,36.20,100.00,1
9,PA,ALTAMIRA,18,516,3018,52.00,LULA,36.20,100.00,1


### 6.2 Distribuição das urnas por % de votos válidos em Lula (2º turno, ≥ 50 válidos)

In [26]:
q('''
    SELECT CAST(floor(100.0 * lula / validos / 5) * 5 AS INTEGER) AS faixa_pct_lula, count(*) AS urnas
    FROM pres WHERE turno = 2 AND validos >= 50
    GROUP BY 1 ORDER BY 1
''').set_index('faixa_pct_lula').T

faixa_pct_lula,0,5,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100
urnas,39,329,1481,5301,13427,24363,37667,49751,57684,55905,50629,43229,38978,32235,27590,24056,18117,10946,5094,1619,88


### 6.3 Urnas com comparecimento ≥ 97% (≥ 50 aptos)

In [27]:
display(q('''
    SELECT turno,
           count(*) FILTER (WHERE comparecimento >= 0.97 * aptos) AS urnas_97pct,
           count(*) FILTER (WHERE comparecimento = aptos)         AS urnas_100pct,
           sum(comparecimento) FILTER (WHERE comparecimento >= 0.97 * aptos) AS votos_nessas_urnas,
           round(100.0 * sum(lula) FILTER (WHERE comparecimento >= 0.97 * aptos)
                 / sum(validos) FILTER (WHERE comparecimento >= 0.97 * aptos), 1) AS pct_lula_nessas_urnas,
           round(100.0 * sum(lula) / sum(validos), 1) AS pct_lula_geral
    FROM pres WHERE aptos >= 50 GROUP BY turno ORDER BY turno
'''))
alto_comparecimento = q('''
    SELECT p.turno, p.uf, s.municipio, p.zona, p.secao, p.aptos, p.comparecimento,
           round(100.0 * p.comparecimento / p.aptos, 1) AS pct_comparecimento,
           p.lula, p.flavio, p.validos, round(100.0 * p.lula / p.validos, 1) AS pct_lula
    FROM pres p JOIN (SELECT 1 AS turno, uf, cd_municipio, zona, secao, municipio FROM secoes_t1 UNION ALL
                      SELECT 2 AS turno, uf, cd_municipio, zona, secao, municipio FROM secoes_t2) s USING (turno, uf, cd_municipio, zona, secao)
    WHERE p.aptos >= 50 AND p.comparecimento >= 0.97 * p.aptos
    ORDER BY pct_comparecimento DESC, p.validos DESC
''')
alto_comparecimento

,turno,urnas_97pct,urnas_100pct,votos_nessas_urnas,pct_lula_nessas_urnas,pct_lula_geral
0,1,124,10,"15,365.00",71.50,45.20
1,2,142,9,"19,843.00",74.70,51.80


,turno,uf,municipio,zona,secao,aptos,comparecimento,pct_comparecimento,lula,flavio,validos,pct_lula
0,2,RN,CARAÚBAS,36,69,163,163,100.00,134.00,25.00,159.00,84.30
1,1,BA,SANTO ESTEVÃO,143,230,168,168,100.00,138.00,9.00,155.00,89.00
2,2,CE,DEPUTADO IRAPUAN PINHEIRO,55,160,127,127,100.00,104.00,21.00,125.00,83.20
3,1,CE,DEPUTADO IRAPUAN PINHEIRO,55,160,127,127,100.00,98.00,20.00,122.00,80.30
4,1,PA,SALVATERRA,3,153,120,120,100.00,89.00,26.00,117.00,76.10
...,...,...,...,...,...,...,...,...,...,...,...,...
261,2,BA,PÉ DE SERRA,114,269,134,130,97.00,98.00,27.00,125.00,78.40
262,2,MG,SÃO SEBASTIÃO DO OESTE,139,105,101,98,97.00,47.00,46.00,93.00,50.50
263,1,RJ,RESENDE,31,365,67,65,97.00,18.00,42.00,64.00,28.10
264,2,PE,SANTA CRUZ DO CAPIBARIBE,109,182,67,65,97.00,44.00,19.00,63.00,69.80


### 6.4 Comparecimento × % de Lula por urna (2º turno, ≥ 50 aptos, sem exterior)

Inclusão de votos em massa gera correlação positiva e uma “cauda” de urnas indo para o canto
(comparecimento 100%, voto 100%). A tabela é um histograma 2D: linhas = faixa de comparecimento, colunas = faixa de % Lula.

In [28]:
fp = q('''
    SELECT uf, 1.0 * comparecimento / aptos AS comparecimento, 1.0 * lula / validos AS pct_lula
    FROM pres WHERE turno = 2 AND aptos >= 50 AND validos > 0 AND uf <> 'ZZ'
''')
print(f"Correlação nacional comparecimento × % Lula: {fp.comparecimento.corr(fp.pct_lula):.3f}")
display(fp.groupby('uf').apply(lambda g: g.comparecimento.corr(g.pct_lula)).round(3).rename('correlacao').to_frame().T)

faixas_c = pd.cut(fp.comparecimento, bins=[i / 100 for i in range(50, 101, 5)], include_lowest=True)
faixas_v = pd.cut(fp.pct_lula, bins=[i / 10 for i in range(0, 11)], include_lowest=True)
pd.crosstab(faixas_c, faixas_v).rename_axis(index='comparecimento', columns='% Lula')

Correlação nacional comparecimento × % Lula: 0.049


uf,AC,AL,AM,AP,BA,CE,DF,ES,GO,MA,MG,MS,MT,PA,PB,PE,PI,PR,RJ,RN,RO,RR,RS,SC,SE,SP,TO
correlacao,-0.21,-0.36,-0.35,-0.26,0.07,-0.26,0.01,-0.17,-0.23,-0.26,-0.22,-0.25,0.01,0.10,-0.30,-0.33,-0.07,-0.25,-0.17,-0.04,0.11,0.13,-0.47,-0.25,-0.26,-0.13,-0.11


% Lula,"(-0.001, 0.1]","(0.1, 0.2]","(0.2, 0.3]","(0.3, 0.4]","(0.4, 0.5]","(0.5, 0.6]","(0.6, 0.7]","(0.7, 0.8]","(0.8, 0.9]","(0.9, 1.0]"
comparecimento,,,,,,,,,,
"(0.499, 0.55]",3,5,8,19,37,54,38,44,33,10
"(0.55, 0.6]",2,14,36,88,142,166,141,147,84,32
"(0.6, 0.65]",7,45,159,314,548,635,548,460,246,103
"(0.65, 0.7]",7,146,612,1606,2918,2759,1942,1355,806,296
"(0.7, 0.75]",31,432,2573,7555,13332,10666,6530,4017,2455,689
"(0.75, 0.8]",42,1177,8727,24874,34604,25192,15361,10121,6002,1438
"(0.8, 0.85]",63,2325,15820,38610,46120,33226,23526,18854,10037,1899
"(0.85, 0.9]",128,2090,8889,14085,15949,16736,18743,14450,7461,1560
"(0.9, 0.95]",80,577,884,794,1058,2646,3671,2095,1483,619


### 6.5 Último dígito das contagens de votos (Lula e Flávio, contagens ≥ 10) — esperado ≈ 10% cada

In [29]:
q('''
    SELECT turno, votos % 10 AS digito, count(*) AS n
    FROM (SELECT turno, lula AS votos FROM pres UNION ALL SELECT turno, flavio FROM pres)
    WHERE votos >= 10
    GROUP BY ALL
''').pivot(index='turno', columns='digito', values='n').apply(lambda r: (100 * r / r.sum()).round(2), axis=1)

digito,0.00,1.00,2.00,3.00,4.00,5.00,6.00,7.00,8.00,9.00
turno,,,,,,,,,,
1,10.07,9.97,9.96,10.05,10.01,10.01,10.02,9.97,9.98,9.97
2,10.02,9.97,9.99,10.04,9.99,10.00,10.01,10.00,9.99,9.98


### 6.6 Variação de comparecimento 1º→2º turno × mudança de voto

`lula_vs_flavio_1t` = % de Lula no confronto direto com Flávio no 1º turno, comparável ao 2º turno.

In [30]:
q(f'''
    WITH a AS (SELECT * FROM pres WHERE turno = 1), b AS (SELECT * FROM pres WHERE turno = 2)
    SELECT CASE WHEN abs(b.comparecimento - a.comparecimento) > {LIMIAR_VARIACAO} * a.comparecimento
                THEN 'variação > 10%' ELSE 'demais urnas' END AS grupo,
           count(*) AS urnas,
           round(100.0 * sum(a.lula) / sum(a.lula + a.flavio), 2) AS lula_vs_flavio_1t,
           round(100.0 * sum(b.lula) / sum(b.validos), 2) AS lula_2t,
           round(100.0 * sum(b.lula) / sum(b.validos) - 100.0 * sum(a.lula) / sum(a.lula + a.flavio), 2) AS ganho_lula_pp
    FROM a JOIN b USING (uf, cd_municipio, zona, secao)
    WHERE a.comparecimento > 0
    GROUP BY 1
''')

,grupo,urnas,lula_vs_flavio_1t,lula_2t,ganho_lula_pp
0,demais urnas,486625,48.77,51.57,2.80
1,variação > 10%,12580,58.11,60.94,2.83


In [31]:
print('Cidades com queda/alta de comparecimento > 15%: variação de voto acompanha o resto do país?')
q('''
    WITH a AS (SELECT uf, cd_municipio, sum(comparecimento) c1, sum(lula) l1, sum(flavio) f1
               FROM pres WHERE turno = 1 GROUP BY ALL),
         b AS (SELECT uf, cd_municipio, sum(comparecimento) c2, sum(lula) l2, sum(validos) v2
               FROM pres WHERE turno = 2 GROUP BY ALL)
    SELECT uf, m.municipio, c1 AS votos_t1, c2 AS votos_t2,
           round(100.0 * (c2 - c1) / c1, 1) AS variacao_pct,
           round(100.0 * l1 / (l1 + f1), 1) AS lula_vs_flavio_1t,
           round(100.0 * l2 / v2, 1) AS lula_2t,
           round(100.0 * l2 / v2 - 100.0 * l1 / (l1 + f1), 1) AS ganho_lula_pp
    FROM a JOIN b USING (uf, cd_municipio)
    JOIN (SELECT DISTINCT uf, cd_municipio, municipio FROM secoes_t1) m USING (uf, cd_municipio)
    WHERE abs(c2 - c1) > 0.15 * c1
    ORDER BY variacao_pct
''')

Cidades com queda/alta de comparecimento > 15%: variação de voto acompanha o resto do país?


,uf,municipio,votos_t1,votos_t2,variacao_pct,lula_vs_flavio_1t,lula_2t,ganho_lula_pp
0,AP,ITAUBAL,"7,354.00","5,239.00",-28.80,54.00,57.00,3.00
1,AP,CUTIAS,"5,191.00","3,758.00",-27.60,55.50,60.10,4.70
2,AC,PORTO WALTER,"6,520.00","4,792.00",-26.50,53.60,53.30,-0.30
3,AC,MARECHAL THAUMATURGO,"9,352.00","7,051.00",-24.60,38.50,40.20,1.70
4,AP,PRACUÚBA,"3,313.00","2,510.00",-24.20,52.30,59.70,7.40
5,AP,FERREIRA GOMES,"6,260.00","4,764.00",-23.90,49.10,52.10,3.00
6,AP,AMAPÁ,"6,262.00","4,971.00",-20.60,59.70,63.40,3.60
7,AP,TARTARUGALZINHO,"9,274.00","7,484.00",-19.30,55.30,58.40,3.10
8,RR,AMAJARI,"6,520.00","5,302.00",-18.70,23.50,27.50,4.00
9,AP,MAZAGÃO,"18,989.00","15,596.00",-17.90,63.40,64.80,1.40


### 6.7 Urnas substituídas entre os turnos (número de série diferente)

In [32]:
q('''
    WITH a AS (SELECT * FROM pres WHERE turno = 1), b AS (SELECT * FROM pres WHERE turno = 2)
    SELECT a.nr_urna <> b.nr_urna AS urna_substituida, count(*) AS urnas,
           round(100.0 * sum(a.lula) / sum(a.lula + a.flavio), 2) AS lula_vs_flavio_1t,
           round(100.0 * sum(b.lula) / sum(b.validos), 2) AS lula_2t
    FROM a JOIN b USING (uf, cd_municipio, zona, secao)
    GROUP BY 1
''')

,urna_substituida,urnas,lula_vs_flavio_1t,lula_2t
0,False,494770,48.99,51.78
1,True,4435,49.22,52.01


### Conclusão dos testes

Com os dados de 2026, nenhum teste apontou padrão de manipulação em larga escala:
- as urnas de candidato único ficam em locais de votação inteiros com voto homogêneo, não em urnas isoladas;
- a distribuição por urna é suave, sem pico artificial;
- comparecimento e % do candidato não têm correlação positiva (na maioria das UFs ela é negativa);
- o último dígito é uniforme;
- urnas com grande variação de comparecimento e urnas substituídas votaram como as demais.

Casos individuais, como as urnas com queda de comparecimento acima de 75%, só podem ser esclarecidos pelo TRE
(por exemplo, pela ata da seção).

## 7. Exportação

In [33]:
for nome, df in {'candidato_unico': unico, 'variacao_cidades': var_cidade,
                 'cidades_variacao_acima_10pct': cidades_10,
                 'urnas_variacao_acima_10pct': urnas_10,
                 'candidato_unico_vs_local': unico_local,
                 'urnas_comparecimento_97pct': alto_comparecimento}.items():
    df.to_csv(SAIDA / f'{nome}.csv', index=False, sep=';', encoding='utf-8-sig')
    print(f'{SAIDA / nome}.csv  ({len(df):,} linhas)')

/Users/alexjunq/Downloads/urnas/resultados/candidato_unico.csv  (204 linhas)
/Users/alexjunq/Downloads/urnas/resultados/variacao_cidades.csv  (5,716 linhas)
/Users/alexjunq/Downloads/urnas/resultados/cidades_variacao_acima_10pct.csv  (114 linhas)
/Users/alexjunq/Downloads/urnas/resultados/urnas_variacao_acima_10pct.csv  (12,580 linhas)
/Users/alexjunq/Downloads/urnas/resultados/candidato_unico_vs_local.csv  (107 linhas)
/Users/alexjunq/Downloads/urnas/resultados/urnas_comparecimento_97pct.csv  (266 linhas)
